# Data Collection

In [1]:
import pandas as pd
import numpy as np
import requests
import concurrent.futures

from tqdm.auto import tqdm

### Quick inspect of the base dataset

In [2]:
BASE_DATA_PATH = '../data/raw/water_quality_training_dataset.csv'

df_base = pd.read_csv(BASE_DATA_PATH)
df_base.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [18]:
df_base.shape

(9319, 6)

In [9]:
df_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9319 entries, 0 to 9318
Data columns (total 6 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Latitude                       9319 non-null   float64
 1   Longitude                      9319 non-null   float64
 2   Sample Date                    9319 non-null   object 
 3   Total Alkalinity               9319 non-null   float64
 4   Electrical Conductance         9319 non-null   float64
 5   Dissolved Reactive Phosphorus  9319 non-null   float64
dtypes: float64(5), object(1)
memory usage: 437.0+ KB


In [7]:
df_base.describe()

,Latitude,Longitude,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
count,9319.000000,9319.000000,9319.000000,9319.000000,9319.000000
mean,-28.474988,26.868414,119.108208,485.004146,43.525338
std,2.760282,3.535164,74.692591,341.937736,50.980194
min,-34.405833,17.730278,4.800000,15.120000,5.000000
25%,-30.160091,26.126667,55.811000,207.050000,10.000000
50%,-28.058889,27.409060,113.300000,402.000000,20.000000
75%,-26.861111,29.245556,170.230000,693.000000,48.000000
max,-22.225556,32.325000,361.676000,1506.000000,195.000000


In [16]:
df_base['Sample Date'] = pd.to_datetime(df_base['Sample Date'], format='%d-%m-%Y')
df_base['Sample Date'].sort_values()

0      2011-01-02
1      2011-01-03
2      2011-01-03
3      2011-01-03
4      2011-01-03
          ...    
9317   2015-12-23
9312   2015-12-23
9316   2015-12-23
5622   2015-12-31
9318   2015-12-31
Name: Sample Date, Length: 9319, dtype: datetime64[ns]

## Static Foundation Data

In [3]:
df = df_base.copy()

Spatial Deduplication

In [4]:
unique_coords = df[['Latitude', 'Longitude']].drop_duplicates().values.tolist()
print(f"Total rows: {len(df)}. Unique locations to process: {len(unique_coords)}")

Total rows: 9319. Unique locations to process: 162


Elevation

In [5]:
elevation_cache = {}

In [6]:
def fetch_elevation_for_coord(coord):
    lat, lon = coord
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
    try:
        # Added a tiny timeout so one bad request doesn't freeze the thread
        response = requests.get(url, timeout=5) 
        if response.status_code == 200:
            return (lat, lon, response.json()['elevation'][0])
    except Exception:
        pass
    return (lat, lon, np.nan)

In [7]:
print("Fetching elevation data using multithreading...")
# ThreadPoolExecutor runs multiple requests at the exact same time
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    # We map our function to our unique coordinates
    results = list(tqdm(executor.map(fetch_elevation_for_coord, unique_coords), total=len(unique_coords)))

# Store the results in our cache
for lat, lon, elev in results:
    elevation_cache[(lat, lon)] = elev

Fetching elevation data using multithreading...


  0%|          | 0/162 [00:00<?, ?it/s]

In [8]:
df['Elevation_m'] = df.apply(lambda row: elevation_cache.get((row['Latitude'], row['Longitude']), np.nan), axis=1)

print("Elevation extraction complete!")

Elevation extraction complete!


Soil Chemistry

In [9]:
def fetch_soil_for_coord(coord):
    """
    Worker function to fetch soil pH and clay content for a single coordinate.
    Returns a tuple: (lat, lon, soil_ph, soil_clay)
    """
    lat, lon = coord
    url = f"https://rest.isric.org/soilgrids/v2.0/properties/query?lon={lon}&lat={lat}&property=phh2o&property=clay&depth=0-5cm"
    
    # Default to NaN in case of failure
    soil_ph = np.nan
    soil_clay = np.nan
    
    try:
        # 10-second timeout is crucial here. If the server hangs, we don't want the thread stuck forever.
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            # Safely navigate the nested JSON
            properties = data.get('properties', {}).get('layers', [])
            
            for prop in properties:
                name = prop.get('name')
                depths = prop.get('depths', [])
                
                if depths:
                    values = depths[0].get('values', {})
                    mean_val = values.get('mean')
                    
                    if mean_val is not None:
                        if name == 'phh2o':
                            soil_ph = mean_val / 10.0 # SoilGrids scales pH by 10
                        elif name == 'clay':
                            soil_clay = mean_val
                            
        return (lat, lon, soil_ph, soil_clay)
        
    except requests.exceptions.RequestException:
        # This catches timeouts, connection resets, etc., allowing the thread to fail gracefully
        return (lat, lon, np.nan, np.nan)
    except Exception:
        # Catch any JSON parsing errors if the API structure changes unexpectedly
        return (lat, lon, np.nan, np.nan)

In [10]:
soil_cache = {}
print("Fetching SoilGrids data using multithreading...")

# We use 5 workers instead of 10 to be polite to the ISRIC servers and avoid HTTP 429 (Too Many Requests)
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    # Map the function to our unique coordinates and wrap in tqdm for a progress bar
    results = list(tqdm(executor.map(fetch_soil_for_coord, unique_coords), total=len(unique_coords)))

# Store the successful results in our dictionary cache
for lat, lon, ph, clay in results:
    soil_cache[(lat, lon)] = {'soil_ph': ph, 'soil_clay_g_kg': clay}

Fetching SoilGrids data using multithreading...


  0%|          | 0/162 [00:00<?, ?it/s]

In [11]:
print("Mapping soil data back to the main dataset...")

# We use .get() twice: first to find the coord dictionary, then to find the specific metric. 
# If either is missing, it falls back to np.nan.
df['soil_ph'] = df.apply(
    lambda row: soil_cache.get((row['Latitude'], row['Longitude']), {}).get('soil_ph', np.nan), 
    axis=1
)

df['soil_clay_g_kg'] = df.apply(
    lambda row: soil_cache.get((row['Latitude'], row['Longitude']), {}).get('soil_clay_g_kg', np.nan), 
    axis=1
)

print("Soil extraction complete!")

Mapping soil data back to the main dataset...
Soil extraction complete!


## Save Collected Dataset

In [12]:
display(df)

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,Elevation_m,soil_ph,soil_clay_g_kg
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,173.0,8.2,69.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,1524.0,6.5,302.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,1472.0,6.1,260.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,1342.0,6.8,264.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,1356.0,6.7,245.0
...,...,...,...,...,...,...,...,...,...
9314,-27.527500,30.858056,23-12-2015,38.900,134.0,20.0,966.0,5.6,331.0
9315,-26.861111,28.884722,23-12-2015,115.800,388.0,20.0,1524.0,6.5,302.0
9316,-26.984722,26.632278,23-12-2015,104.874,835.0,148.0,NaN,6.6,234.0
9317,-27.935000,26.126667,23-12-2015,128.000,305.0,28.0,1252.0,7.0,207.0


In [13]:
output_path = '../data/processed/elevation_soil_features.csv'
df.to_csv(output_path, index=False)
print(f"Dataset saved to {output_path}")

Dataset saved to ../data/processed/elevation_soil_features.csv
